In [ ]:
import os

import earthaccess
import fiona
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pandarallel import pandarallel

In [ ]:
def specialClip(sj):
    left = gpd.GeoSeries(sj.geometry)
    right = gpd.GeoSeries(sj.geom)
    geometry_clip = left.clip(right)
    return geometry_clip

### Make table of intersecting NHD HUC4 and SWOT tiles

#### HUC Boundaries

In [ ]:
# Read in HUC4 boundaries (no Great Lakes)
data_path = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/all_wbd_no_great_lakes.parquet'
wbd = gpd.read_parquet(path=data_path)

In [ ]:
# Project CRS to WGS 84 / Pseudo-Mercator
wbd_wm = wbd.to_crs(epsg=3857)

#### Make lightweight version of SWOT Science Orbit tiles

In [ ]:
data_path_sci = '/nas/cee-water/cjgleason/data/SWOT/tiles/swot_science_coverage_20240319.kmz'
layers = fiona.listlayers(data_path_sci)

In [ ]:
# Initialize lists
d = []

# Loop through indices and store in lists
for i in range(3,len(layers[3:])+1):

    # Read in the current SWOT pass
    pass_tiles = gpd.read_file(filename=data_path_sci, layer=layers[i], engine='pyogrio')
    
    subset = pass_tiles.iloc[:,[0,-1]]

    # Append to lists
    d.append(subset)

# Merge GeoDataFrames
all_tiles = pd.concat(d)

In [ ]:
# Write out lightweight files as parquet and GeoPackage
save_path = '/nas/cee-water/cjgleason/data/SWOT/tiles/'
all_tiles.to_parquet(path=os.path.join(save_path, 'swot_science_coverage_merged.parquet'))
all_tiles.to_file(filename=os.path.join(save_path, 'swot_science_coverage_merged.gpkg'), driver='GPKG')

#### Join tiles to HUC4 to make reference table

In [ ]:
data_path = '/nas/cee-water/cjgleason/data/SWOT/tiles/swot_science_coverage_merged.parquet'
# Read in simplified SWOT science tiles
sci_tiles = gpd.read_parquet(path=data_path)

# Project CRS to WGS 84 / Pseudo-Mercator
sci_tiles_wm = sci_tiles.to_crs(epsg=3857)

# Get bounding box of NHDPlus HR watersheds
bounds_wbd = wbd_wm.total_bounds

# Clip SWOT science tiles to counding box
sci_tiles_clip = gpd.clip(gdf=sci_tiles_wm, mask=bounds_wbd)

# Copy geometry column for WBD as sjoin will discard it
wbd_wm['geom'] = wbd_wm.geometry

# Join the SWOT tiles and HUC4 boundaries by intersection
joined = gpd.sjoin(sci_tiles_clip, wbd_wm, how='inner', predicate='intersects')
joined = joined.reset_index().rename(columns={'index': 'index_old'})

In [ ]:
pandarallel.initialize()

In [ ]:
joined['geometry_clipped'] = joined.parallel_apply(user_defined_function=specialClip,
                                                   axis=1)
joined['coverage'] = joined.geometry_clipped.area / joined.geometry.area
cleaned = joined[['Name', 'huc4', 'coverage']].sort_values(by=['Name', 'huc4']).rename(columns={'Name': 'tile'}).reset_index().drop(columns='index')
cleaned.to_csv('../data/huc4_swot_science_tiles.csv', index=False)

### Make table of intersecting Pfaf Level 2 (SWORD) and SWOT tiles

#### Pfaf Boundaries

In [ ]:
# Read in HUC4 boundaries (no Great Lakes)
data_path = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/hybas_ar_na_lev01-12_v1c_clipped_to_nhd_no_great_lakes.parquet'
pfaf = gpd.read_parquet(path=data_path)

# Project CRS to WGS 84 / Pseudo-Mercator
pfaf_wm = pfaf.to_crs(epsg=3857)

#### Join tiles to SWORD basins to make reference table

In [ ]:
data_path = '/nas/cee-water/cjgleason/data/SWOT/tiles/swot_science_coverage_merged.parquet'
# Read in simplified SWOT science tiles
sci_tiles = gpd.read_parquet(path=data_path)

# Project CRS to WGS 84 / Pseudo-Mercator
sci_tiles_wm = sci_tiles.to_crs(epsg=3857)

# Get bounding box of NHDPlus HR watersheds
bounds_pfaf = pfaf_wm.total_bounds

# Clip SWOT science tiles to counding box
sci_tiles_clip = gpd.clip(gdf=sci_tiles_wm, mask=bounds_pfaf)

# Copy geometry column for WBD as sjoin will discard it
pfaf_wm['geom'] = pfaf_wm.geometry

# Join the SWOT tiles and HUC4 boundaries by intersection
joined = gpd.sjoin(sci_tiles_clip, pfaf_wm, how='inner', predicate='intersects')
joined = joined.reset_index().rename(columns={'index': 'index_old'})

In [ ]:
pandarallel.initialize()

In [ ]:
joined['geometry_clipped'] = joined.parallel_apply(user_defined_function=specialClip,
                                                   axis=1)
joined['coverage'] = joined.geometry_clipped.area / joined.geometry.area
cleaned = joined[['Name', 'PFAF_ID', 'coverage']].sort_values(by=['Name', 'PFAF_ID']).rename(columns={'Name': 'tile'}).reset_index().drop(columns='index')
cleaned.to_csv('../data/pfaf_level_2_swot_science_tiles.csv', index=False)

### Make table of intersecting USGS gauges and SWOT tiles

#### USGS gauges

In [ ]:
# Table of all gauges with NHDPlus HR reaches mapped
nhd_sites = gpd.read_parquet('/nas/cee-water/cjgleason/fiona/SWOT_narrow_rivers/data/usgs_stations.parquet')

# Subset gauges to those with available stage data
nhd_sites = nhd_sites[nhd_sites['site_id'].isin(gauges_w_data)].reset_index()
nhd_sites = nhd_sites.drop(columns='index')

nhd_sites = nhd_sites[['site_id', 'name', 'geometry']]
nhd_sites = nhd_sites.to_crs(epsg=3857)


#### Join tiles to USGS gauges to make reference table

In [ ]:
data_path = '/nas/cee-ice/data/SWOT/tiles/swot_science_coverage_merged.parquet'
# Read in simplified SWOT science tiles
sci_tiles = gpd.read_parquet(path=data_path)

# Project CRS to WGS 84 / Pseudo-Mercator
sci_tiles_wm = sci_tiles.to_crs(epsg=3857)

# Get bounding box of gauges
bounds_gauges = nhd_sites.total_bounds

# Clip SWOT science tiles to bounding box
sci_tiles_clip = gpd.clip(gdf=sci_tiles_wm, mask=bounds_gauges)

# Copy geometry column for WBD as sjoin will discard it
nhd_sites['geom'] = nhd_sites.geometry

# Join the SWOT tiles and HUC4 boundaries by intersection
joined = gpd.sjoin(sci_tiles_clip, nhd_sites, how='inner', predicate='contains')
joined = joined.reset_index().rename(columns={'index': 'index_old'})

cleaned = joined[['Name', 'site_id']].sort_values(by=['Name', 'site_id']).rename(columns={'Name': 'tile'}).reset_index().drop(columns='index')

cleaned.to_csv('../data/usgs_gauges_swot_science_tiles.csv', index=False)

### Coverage plots

In [ ]:
subset = joined[joined['huc4'] == '0108']

huc = gpd.GeoSeries(subset.iloc[0].geom)
huc4 = subset.iloc[0].huc4

fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(nrows=2, ncols=2, figsize=(8,12))

plt.suptitle('HUC4 ' + str(huc4))

huc.plot(ax=ax0, color='b')
gpd.GeoSeries(subset.geometry).plot(ax=ax0, color='y', alpha=0.5)
ax0.set_title('All tiles')

huc.plot(ax=ax1, color='b')
gpd.GeoSeries(subset[subset['coverage'] > 0.1] .geometry).plot(ax=ax1, color='y', alpha=0.5)
ax1.set_title('Tiles with > 10% overlap')

huc.plot(ax=ax2, color='b')
gpd.GeoSeries(subset[subset['coverage'] > 0.2] .geometry).plot(ax=ax2, color='y', alpha=0.5)
ax2.set_title('Tiles with > 20% overlap')

huc.plot(ax=ax3, color='b')
gpd.GeoSeries(subset[subset['coverage'] > 0.3] .geometry).plot(ax=ax3, color='y', alpha=0.5)
ax3.set_title('Tiles with > 30% overlap')

fig.tight_layout()

In [ ]:
subset = joined[joined['huc4'] == '1606']

huc = gpd.GeoSeries(subset.iloc[0].geom)
huc4 = subset.iloc[0].huc4

fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(nrows=2, ncols=2, figsize=(8,12))

plt.suptitle('HUC4 ' + str(huc4))

huc.plot(ax=ax0, color='b')
gpd.GeoSeries(subset.geometry).plot(ax=ax0, color='y', alpha=0.5)
ax0.set_title('All tiles')

huc.plot(ax=ax1, color='b')
gpd.GeoSeries(subset[subset['coverage'] > 0.1] .geometry).plot(ax=ax1, color='y', alpha=0.5)
ax1.set_title('Tiles with > 10% overlap')

huc.plot(ax=ax2, color='b')
gpd.GeoSeries(subset[subset['coverage'] > 0.2] .geometry).plot(ax=ax2, color='y', alpha=0.5)
ax2.set_title('Tiles with > 20% overlap')

huc.plot(ax=ax3, color='b')
gpd.GeoSeries(subset[subset['coverage'] > 0.3] .geometry).plot(ax=ax3, color='y', alpha=0.5)
ax3.set_title('Tiles with > 30% overlap')

fig.tight_layout()

In [ ]:
subset = joined[joined['huc4'] == '1711']

huc = gpd.GeoSeries(subset.iloc[0].geom)
huc4 = subset.iloc[0].huc4

fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(nrows=2, ncols=2, figsize=(8,12))

plt.suptitle('HUC4 ' + str(huc4))

huc.plot(ax=ax0, color='b')
gpd.GeoSeries(subset.geometry).plot(ax=ax0, color='y', alpha=0.5)
ax0.set_title('All tiles')

huc.plot(ax=ax1, color='b')
gpd.GeoSeries(subset[subset['coverage'] > 0.1] .geometry).plot(ax=ax1, color='y', alpha=0.5)
ax1.set_title('Tiles with > 10% overlap')

huc.plot(ax=ax2, color='b')
gpd.GeoSeries(subset[subset['coverage'] > 0.2] .geometry).plot(ax=ax2, color='y', alpha=0.5)
ax2.set_title('Tiles with > 20% overlap')

huc.plot(ax=ax3, color='b')
gpd.GeoSeries(subset[subset['coverage'] > 0.3] .geometry).plot(ax=ax3, color='y', alpha=0.5)
ax3.set_title('Tiles with > 30% overlap')

fig.tight_layout()

In [ ]:
subset = joined[joined['huc4'] == '0309']

huc = gpd.GeoSeries(subset.iloc[0].geom)
huc4 = subset.iloc[0].huc4

fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(nrows=2, ncols=2, figsize=(8,12))

plt.suptitle('HUC4 ' + str(huc4))

huc.plot(ax=ax0, color='b')
gpd.GeoSeries(subset.geometry).plot(ax=ax0, color='y', alpha=0.5)
ax0.set_title('All tiles')

huc.plot(ax=ax1, color='b')
gpd.GeoSeries(subset[subset['coverage'] > 0.1] .geometry).plot(ax=ax1, color='y', alpha=0.5)
ax1.set_title('Tiles with > 10% overlap')

huc.plot(ax=ax2, color='b')
gpd.GeoSeries(subset[subset['coverage'] > 0.2] .geometry).plot(ax=ax2, color='y', alpha=0.5)
ax2.set_title('Tiles with > 20% overlap')

huc.plot(ax=ax3, color='b')
gpd.GeoSeries(subset[subset['coverage'] > 0.3] .geometry).plot(ax=ax3, color='y', alpha=0.5)
ax3.set_title('Tiles with > 30% overlap')

fig.tight_layout()